# 07 — The Final Push

**Start: LB 0.95006. End: LB 0.95048 (CV 0.95036), rank ≈ 339/2192 — top 15.5%.**

Milestone 6's scouting produced a ranked queue of unexplored edges; this notebook ran the queue. Training happened in `src/` scripts (heavy jobs, cached to `artifacts/`); this notebook is the record and the final combination.

## E9 — CatBoost + exact-value categoricals: the tree-safe TE (`src/train_cat_ev.py`)

The milestone's best idea, and it was born from a *failure*. E6 (TE→LightGBM) lost 4 points because *we* pre-computed target encodings and handed the booster dense, target-flavored noise. But CatBoost's ordered target statistics **are** target encoding — computed internally, per random permutation, with each row encoded only from rows "before" it. So we fed it the raw exact-value categoricals (`ev_steps`, `ev_cal`, `ev_water`, `ev_sleep`, `ev_hr_bmi` — thousands of levels) and let it encode them its own way.

**Result: OOF 0.94982, Δ +0.00059 over its own baseline — the best single model of the project.** The same feature idea moved three function classes in three directions: trees + naive TE (−0.0044), our NN + TE (+0.0001), CatBoost + native TE (+0.0006). Encoding *mechanism* mattered more than the features themselves.

## E10 — RealMLP-TD via pytabkit (`src/train_realmlp.py`)

The published honest frontier (OOF 0.95062) is a RealMLP; ours came out **0.94731** after prior-correction (argmax 0.884 raw — pytabkit has no class weights, so the correction does the metric-alignment work post-hoc). Why short of the published number? CPU budget: 6 epochs vs their tuned schedule, no test-time refit, 5-fold vs 7. The greedy blend ultimately gave it **weight 0** — diversity again failed to beat a ~0.003 quality gap. Second confirmation of milestone 5's law.

## E11 — HGB tuning (`src/tune_hgb.py`): Δ +0.00007, below the bar

Winner `max_leaf_nodes=23, l2=1.0` (the public kernel's smaller-trees intuition confirmed directionally), but the full-CV re-verify stripped the fold-0 proxy's +0.0009 down to noise. Third "defaults were fine" in the project's ledger. The tuned variant still enters the blend pool — at zero marginal cost, let the combiner decide.

## E12 — NN seed averaging (`src/train_nn_te_seed.py`)

Seeds 42/1/2: OOF 0.94831/0.94835/0.94808, averaged → a steadier `nn_te` member (its fold spread was our widest).

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
CLASSES = ['at-risk', 'fit', 'unhealthy']
y = train['health_condition'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
A = '../artifacts/'

def load_members(kind):
    m = {'lgbm': np.mean([np.load(f'{A}{n}_{kind}.npy')
                          for n in ('lgbm_tuned', 'lgbm_seed1', 'lgbm_seed2')], axis=0),
         'cat': np.load(f'{A}cat_{kind}.npy'),
         'cat_ev': np.load(f'{A}cat_ev_{kind}.npy'),
         'hgb': np.load(f'{A}hgb_{kind}.npy'),
         'hgb_tuned': np.load(f'{A}hgb_tuned_{kind}.npy'),
         'nn_te': np.mean([np.load(f'{A}nn_te_{kind}.npy'),
                           np.load(f'{A}nn_te_s1_{kind}.npy'),
                           np.load(f'{A}nn_te_s2_{kind}.npy')], axis=0),
         'realmlp': np.load(f'{A}realmlp_{kind}.npy'),
         'lgbm_te': np.load(f'{A}lgbm_te_{kind}.npy')}
    # prior-correct the unweighted-loss RealMLP so calibrations are comparable
    rm = m['realmlp'] * np.array([1.0, 8.43, 6.44])
    m['realmlp'] = rm / rm.sum(1, keepdims=True)
    return m

oof = load_members('oof')
cls_counts = np.bincount(y)
def bal_acc(pred):
    return float((np.bincount(y[pred == y], minlength=3) / cls_counts).mean())
pd.Series({n: bal_acc(o.argmax(1)) for n, o in oof.items()}).round(5).sort_values(ascending=False)

cat_ev       0.94982
hgb_tuned    0.94963
hgb          0.94956
cat          0.94923
nn_te        0.94865
lgbm         0.94831
realmlp      0.94731
lgbm_te      0.94451
dtype: float64

## The grand blend — greedy ensemble selection

Eight members makes an exhaustive weight grid infeasible (7 free dimensions), so we switch combiners one last time: **greedy selection with replacement** (Caruana et al. 2004). Start empty; 80 rounds; each round, add whichever member most improves the running average's OOF balanced accuracy — repeats allowed, so selection counts become fractional weights. Properties worth teaching: it prices quality *and* diversity implicitly (a redundant model never gets picked), handles any member count, and can't produce negative weights (a regularizer in disguise: negative blend weights on OOF are usually overfit).

In [2]:
names = list(oof)
P = np.stack([oof[n].astype('float32') for n in names])
ROUNDS = 80
counts = np.zeros(len(names), int)
cur = np.zeros_like(P[0])
for r in range(ROUNDS):
    scores = [bal_acc(((cur * counts.sum() + P[i]) / (counts.sum() + 1)).argmax(1))
              for i in range(len(names))]
    i = int(np.argmax(scores))
    counts[i] += 1
    cur = (cur * (counts.sum() - 1) + P[i]) / counts.sum()
w = counts / counts.sum()
print("weights:", {n: round(float(w[i]), 3) for i, n in enumerate(names) if w[i] > 0})
print(f"blend OOF (argmax): {bal_acc(cur.argmax(1)):.5f}")

best = (1.0, 1.0, bal_acc(cur.argmax(1)))
for a in np.arange(0.7, 1.5, 0.02):
    for b in np.arange(0.7, 1.5, 0.02):
        st = bal_acc((cur * np.array([1, a, b], dtype='float32')).argmax(1))
        if st > best[2]: best = (a, b, st)
a, b, cv = best
print(f"+ thresholds({a:.2f}, {b:.2f}) = {cv:.5f}")

weights: {'lgbm': 0.013, 'cat_ev': 0.388, 'hgb': 0.113, 'hgb_tuned': 0.325, 'nn_te': 0.15, 'lgbm_te': 0.013}
blend OOF (argmax): 0.95036


+ thresholds(1.00, 1.00) = 0.95036


In [3]:
testp = load_members('testp')
p = sum(w[i] * testp[n] for i, n in enumerate(names)) * np.array([1.0, a, b])
sub = pd.DataFrame({'id': test['id'],
                    'health_condition': [CLASSES[i] for i in p.argmax(1)]})
sub.to_csv('../submissions/07-grand-blend.csv', index=False)
print(sub['health_condition'].value_counts(normalize=True).round(4))

health_condition
at-risk      0.8111
unhealthy    0.1152
fit          0.0737
Name: proportion, dtype: float64


## Result

| | CV | public LB |
|---|---|---|
| milestone 6 blend | 0.95009 | 0.95006 |
| **grand blend** | **0.95036** | **0.95048** |

Rank ≈ **339 / 2192 (top 15.5%)**. Notable details:

- **Thresholds converged to (1.00, 1.00)** — the first time in the project the decision rule needed no correction. A committee of individually metric-aligned members is collectively aligned; the alignment job migrated from post-processing into the members themselves.
- **The weight vector is the milestone in miniature:** `cat_ev 0.39` (E9's tree-safe TE), `hgb + hgb_tuned 0.44`, `nn_te 0.15` — while `realmlp` (too weak even prior-corrected) and plain `cat` (strictly dominated by `cat_ev`) got zero. Greedy selection did the pruning no human argument could settle.
- **CV−LB = −0.0001**: seven submissions, and the instrument never drifted.

## Project close

Final arc: **0.333 → 0.875 → 0.949 → 0.9497 → 0.9498 → 0.9501 → 0.9505.** Every increment after the second came from alignment, diversity, or a measured redirect after a falsified hypothesis — never from a lucky guess. What's left above us on the public board is, per the milestone-6 scan, substantially shared-file blending with documented private-split fragility; our stack is honest OOF end to end. The private leaderboard resolves after July 31.